# 🎵 Análisis de hábitos de escucha musical en Springfield y Shelbyville

## Introducción

Las plataformas de streaming musical generan información valiosa sobre los hábitos de consumo de sus usuarios. En este proyecto se analizan datos de reproducción musical de las ciudades de Springfield y Shelbyville con el objetivo de identificar posibles diferencias en la actividad de escucha según la ubicación y el día de la semana.

A lo largo del análisis se realizará una etapa de limpieza y preparación de datos para garantizar la calidad de la información, seguida de un análisis exploratorio que permita evaluar distintos patrones de comportamiento entre ambas ciudades.

La pregunta principal que guía este estudio es:

**¿Existen diferencias en la actividad de escucha musical entre Springfield y Shelbyville dependiendo del día de la semana?**

## Descripción de los datos

El conjunto de datos contiene información sobre reproducciones musicales realizadas por usuarios de dos ciudades distintas.

Las variables disponibles incluyen:

* **userID:** identificador único del usuario.
* **Track:** nombre de la canción reproducida.
* **artist:** artista de la canción.
* **genre:** género musical.
* **City:** ciudad del usuario.
* **time:** hora de reproducción.
* **Day:** día de la semana.

Esta información permitirá analizar patrones de consumo musical y comparar el comportamiento de los usuarios entre ambas ciudades.


## Preparación y limpieza de datos

Antes de realizar cualquier análisis es necesario evaluar la calidad de la información disponible.

Durante esta etapa se revisarán posibles inconsistencias en los nombres de las columnas, valores ausentes, registros duplicados y errores de escritura que puedan afectar los resultados del análisis.

In [1]:
# Importación de librería "pandas"
import pandas as pd

In [2]:
# Lectura del dataset y almacenamiento en la variable df
df = pd.read_csv("data/music_project_en.csv")

# Visualización del contenido de la tabla df
print(df.head(10))


     userID                        Track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

        City        time        Day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

In [3]:
# Visualización de la nformación general sobre nuestros datos
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63736 non-null  object
 2   artist    57512 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


## Comprensión inicial de los datos

La revisión preliminar permitió identificar la estructura general del conjunto de datos y las variables disponibles para el análisis.

Durante esta etapa se detectaron algunos problemas de calidad de datos, incluyendo **inconsistencias en los nombres de las columnas, valores ausentes y registros duplicados.** Estos hallazgos hicieron necesaria una fase de preprocesamiento antes de iniciar el análisis exploratorio.

## Preprocesamiento de los datos

Con base en la visualización previa el objetivo aquí es preparar los datos para analizarlos; resolviendo inconsistencias con los encabezados, valores ausentes y duplicados

### Estilo del encabezado

In [4]:
# Vista de los nombres de las columnas
df.columns

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')

Cambiaremos los encabezados de la tabla siguiendo las reglas estilísticas convencionales:
*   Todos los caracteres deben ser minúsculas.
*   Eliminar los espacios.
*   Si el nombre tiene varias palabras, utilizar snake_case, es decir, añadir un guion bajo ( _ ) entre las palabras en lugar de un espacio.


Utilizaremos un **bucle for** para iterar sobre los nombres de las columnas y poner todos los caracteres en minúsculas.

In [5]:
# Bucle que itera sobre los encabezados y los pone todos en minúsculas
columnas_min =[]

for minus in df.columns:
    col_min = minus.lower()
    columnas_min.append(col_min)
    
df.columns = columnas_min
print(df.columns)

Index(['  userid', 'track', 'artist', 'genre', '  city  ', 'time', 'day'], dtype='object')


Utilizando el mismo método, eliminaremos los espacios al principio y al final de los nombres de las columnas.

In [6]:
# Bucle que itera sobre los encabezados y elimina los espacios
columnas_sin_espacios = []
for espacios in df.columns:
    nombre_sin_espacios = espacios.strip()
    columnas_sin_espacios.append(nombre_sin_espacios)

df.columns= columnas_sin_espacios
print(df.columns)

Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


Aplicaremos la regla de snake_case en la columna `userid` quedando como `user_id`

In [7]:
df = df.rename(columns={"userid": "user_id"})

# Comprobamos que los cambios se hayan realizado correctamente
print(df.columns) 


Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')


### Valores ausentes
 Buscaremos los valores ausentes de la tabla utilizando el método **isna()**

In [8]:
# Cálculo de valores ausentes
print('Número de valores ausentes:')
df.isna().sum() 


Número de valores ausentes:


user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64

Sustituiremos los valores ausentes en las columnas `'track'`, `'artist'` y `'genre'` con el string `'unknown'`.

1. Crearemos una lista llamada columns_to_replace que contenga los nombres de las columnas 'track', 'artist' y 'genre'.

2. Con un bucle for iteraremos sobre cada columna en columns_to_replace.

3. Dentro del bucle, se sustituirá los valores ausentes en cada columna con el string `'unknown'`.

In [9]:
# Bucle para iterar en los encabezados reemplazando los valores ausentes con 'unknown'
columns_to_replace = ["track", "artist", "genre"] 
for ausentes in columns_to_replace:
    df[ausentes].fillna("unknown", inplace=True)
    

Para asegurarnos de que no falten valores ausentes por reemplazar en el conjunto de datos, contaremos los valores ausentes una vez más.

In [10]:
print('Número de valores ausentes:')
df.isna().sum() 


Número de valores ausentes:


user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64

### Duplicados
Localizar si hay duplicados duplicados explícitos en la tabla utilizando el método **duplicated()**

In [11]:
# Contar los duplicados explícitos
df.duplicated().sum()

np.int64(3826)

Se han registrado 3,826 valores duplicados por lo que procedemos a eliminarlos llamando al método **drop_duplicates()**.

In [12]:
 # Eliminación de duplicados explícitos
df.drop_duplicates(inplace=True)

# Corrobación de que se hayan eliminado los duplicados
df.duplicated().sum()

np.int64(0)

Validaremos si hay **duplicados implícitos** específicamente en la columna `genre`. Por ejemplo, el nombre de un género se puede escribir de varias formas por lo que estos errores también pueden afectar al resultado.

1. Utilizaremos difflib para localizar cadenas de texto similares entre si.
2. Por medio de un bucle iteraremos para buscar coincidencias cercanas en la lista 


2. Llamaremos al método que devolverá todos los valores únicos en la columna extraída.


In [13]:
# Importacipon de librería
import difflib

generos_unicos = sorted(df['genre'].dropna().unique())
duplicados_encontrados = {}

for genero in generos_unicos:
    #Buscamos coincidencias cercanas en la lista, descartando coincidir exactamente cosigo mismo
    coincidencias = difflib.get_close_matches(genero, generos_unicos, n=5, cutoff=0.8)

    #Si en la iteración se encontró otros géneros parecidos, los guardamos
    if len(coincidencias) > 1:
        duplicados_encontrados[genero] = coincidencias

# Visualizar el resultado
for genero, variantes in duplicados_encontrados.items():
    print(f'Género: {genero} -> Posibles duplicados: {variantes}')

Género: classical -> Posibles duplicados: ['classical', 'classicmetal']
Género: classicmetal -> Posibles duplicados: ['classicmetal', 'classical']
Género: cuban -> Posibles duplicados: ['cuban', 'urban']
Género: eastern -> Posibles duplicados: ['eastern', 'western']
Género: electronic -> Posibles duplicados: ['electronic', 'popelectronic']
Género: hardcore -> Posibles duplicados: ['hardcore', 'posthardcore']
Género: hip-hop -> Posibles duplicados: ['hip-hop', 'hiphop']
Género: hiphop -> Posibles duplicados: ['hiphop', 'hip-hop']
Género: jazz -> Posibles duplicados: ['jazz', 'nujazz']
Género: jpop -> Posibles duplicados: ['jpop', 'pop']
Género: latin -> Posibles duplicados: ['latin', 'latino']
Género: latino -> Posibles duplicados: ['latino', 'latin']
Género: local -> Posibles duplicados: ['local', 'vocal']
Género: metal -> Posibles duplicados: ['metal', 'numetal']
Género: nujazz -> Posibles duplicados: ['nujazz', 'jazz']
Género: numetal -> Posibles duplicados: ['numetal', 'metal']
Géne

En la lista se identificaron **duplicados implícitos** del género `hiphop`, es decir, nombres mal escritos o variantes que hacen referencia al mismo género musical como: `hip`, `hop`, `hip-hop`  

Para solucionarlo, vamos a crear una función llamada `replace_wrong_values()` que reciba los siguientes parámetros:

* `df`: el DataFrame a modificar
* `column`: el nombre de la columna a trabajar
* `wrong_values`: una lista con los valores incorrectos
* `correct_value`: el valor correcto para reemplazar

Dentro de la función, usaremos un bucle `for` para iterar sobre cada valor incorrecto y aplicar `.replace()`.



In [14]:
# Función para reemplazar los duplicados implícitos
def replace_wrong_values(df, column, wrong_values, correct_value):
    for valor_incorrecto in wrong_values:
        df[column] = df[column].replace(valor_incorrecto, correct_value)
    return df
incorrectos = ["hip", "hop", "hip-hop"]
correcto = "hiphop"

Ahora llamaremos a la función pasando:

* `df` como el DataFrame
* `'genre'` como nombre de columna
* `['hip', 'hop', 'hip-hop']` como lista de valores incorrectos
* `'hiphop'` como valor correcto


In [15]:
# Eliminación de duplicados implícitos
df = replace_wrong_values(df, "genre", incorrectos, correcto)

Nos asegúramos de que los nombres duplicados de hiphop se hayan eliminado.

In [16]:
# Comprobación de duplicados implícitos
print(sorted(df["genre"].unique()))

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

### Resultados de la limpieza

Antes de analizar los hábitos de escucha, se realizaron tareas de limpieza y estandarización para garantizar la consistencia de la información.

Las actividades incluyeron:

- Corrección de nombres de columnas.
- Tratamiento de valores ausentes.
- Eliminación de registros duplicados.
- Homologación de categorías con errores de escritura.

Estas acciones permiten reducir sesgos y mejorar la confiabilidad de los resultados.

## Análisis Exploratorio


### Comparación de la actividad musical por ciudad

Una vez completada la preparación de los datos, se analizará la actividad de escucha en Springfield y Shelbyville para identificar posibles diferencias entre ambas ciudades.

El objetivo es determinar si la ubicación geográfica influye en la cantidad de reproducciones registradas y en los patrones de consumo musical observados.

**Actividad de escucha según el día de la semana**

Además de comparar las ciudades, resulta relevante evaluar cómo varía la actividad musical a lo largo de la semana.

Este análisis permitirá identificar si existen días con una mayor participación de los usuarios y si dichas tendencias son consistentes entre Springfield y Shelbyville.

Queremos analizar si hay diferencias en la cantidad de canciones reproducidas en Springfield y Shelbyville. Para ello, usaremos los datos de dos días de la semana: lunes y viernes.

Compararemos cuántas canciones se escucharon en cada ciudad durante esos días para identificar posibles patrones de comportamiento.

- Dividiremos los datos, agrupando por ciudad.

- Buscaremos cuántas canciones se reproducen en cada grupo.

### ¿Cuántas canciones se reprodujeron en cada ciudad?

Utilizaremos la columna `track` como referencia para contar el número de canciones reproducidas en cada ciudad.

In [17]:
# Número de canciones reproducidas en cada ciudad
ciudades= df.groupby("city")["track"].count()
print(ciudades)

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64


### Comparación inicial entre ciudades

Los resultados muestran una diferencia considerable en la actividad musical entre ambas ciudades.

Springfield registra un volumen significativamente mayor de reproducciones en comparación con Shelbyville, con una diferencia superior a 24 mil 229 canciones durante el periodo analizado.

Esta diferencia podría estar relacionada con factores externos como el tamaño de la población, el nivel de adopción de la plataforma o distintos hábitos de consumo musical entre los usuarios de cada ciudad.

### ¿Cuánta canciones se reprodujeron los lunes y viernes?

Agruparemos los datos por día de la semana y cuontaremos cuántas canciones se reprodujeron los lunes y viernes.



In [18]:
# Cálculo de las canciones reproducidas en cada uno de los dos días
reproducciones_por_dia = df.groupby("day")["track"].count()
print(reproducciones_por_dia)


day
Friday       21840
Monday       21354
Wednesday    18059
Name: track, dtype: int64


Etapa 3.4. Describe brevemente qué observaste al comparar los lunes y viernes.

¿Hubo un día con más actividad? ¿Cambia algo si analizas cada ciudad por separado?

### Actividad de escucha por día

El análisis muestra que el viernes es el día con mayor número de reproducciones, seguido por el lunes.

Este comportamiento podría estar relacionado con cambios en las rutinas de los usuarios al acercarse el fin de semana. Sin embargo, para comprender mejor este patrón es necesario analizar los resultados por ciudad y determinar si la tendencia se mantiene de forma consistente en ambos grupos.

Ahora combinaremos dos criterios: día y ciudad. Creando una función llamada `number_tracks()` que reciba dos parámetros:

* `day`: un día de la semana (por ejemplo, `'Monday'`)
* `city`: el nombre de una ciudad (por ejemplo, `'Springfield'`)

Dentro de la función:

1. Filtrar el DataFrame por el día.
2. Filtrar por la ciudad.
3. Contar cuántas veces aparece `'user_id'` en ese filtro.


In [19]:
# Declaramos la función number_tracks() con dos parámetros: day= y city=.
def number_tracks(day, city):
    # Almacenamos las filas del DataFrame donde el valor en la columna 'day' es igual al parámetro day=
    dia = df[df["day"] == day]
    # Filtro para las filas donde el valor en la columna 'city' es igual al parámetro city=
    ciudad = dia[dia["city"] == city]
    # Extraemos la columna 'user_id' de la tabla filtrada y aplicamos el método count()
    usuario = ciudad["user_id"].count()
    # Devolvemos el número de valores de la columna 'user_id'
    return usuario

Llamaremos a `number_tracks()` cuatro veces, una por ciudad en cada uno de los dos días.

In [20]:
print('Número de canciones reproducidas el día Lunes:')

springfield_lun = number_tracks("Monday", "Springfield")
print(f'Springfield: {springfield_lun}')

shelbyville_lun = number_tracks("Monday", "Shelbyville")
print(f'Shelbyville: {shelbyville_lun}')

print()
print('Número de canciones reproducidas el día Viernes:')

springfield_vie = number_tracks("Friday", "Springfield")
print(f'Springfield: {springfield_vie}')

shelbyville_vie = number_tracks("Friday", "Shelbyville")
print(f'Shelbyville: {shelbyville_vie}')



Número de canciones reproducidas el día Lunes:
Springfield: 15740
Shelbyville: 5614

Número de canciones reproducidas el día Viernes:
Springfield: 15945
Shelbyville: 5895


## Conclusiones

Este proyecto permitió explorar los hábitos de escucha musical de usuarios en Springfield y Shelbyville mediante un proceso que incluyó limpieza, preparación y análisis de datos.

Durante la etapa de preprocesamiento se corrigieron inconsistencias en los nombres de las columnas, se trataron valores ausentes, se eliminaron registros duplicados y se estandarizaron categorías con errores de escritura. Estas acciones permitieron mejorar la calidad y confiabilidad de la información analizada.

Los resultados muestran diferencias en el volumen total de reproducciones entre ambas ciudades, siendo Springfield la que registra una mayor actividad. Asimismo, se identificaron variaciones en la cantidad de reproducciones según el día de la semana, lo que sugiere que los hábitos de escucha no son completamente uniformes a lo largo del tiempo.

En conjunto, el análisis permitió responder la pregunta planteada inicialmente y demostró la importancia de la preparación de datos como etapa fundamental para obtener conclusiones confiables y respaldadas por evidencia.
